# SecureCode AI — Full-Scale Cross-Dataset Evaluation (E1–E4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0Tusharbhardwaj/nlp/blob/main/notebooks/04_cross_dataset_evaluation.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-0Tusharbhardwaj%2Fnlp-blue.svg)](https://github.com/0Tusharbhardwaj/nlp)

## 1. Scientific Objective
Execute the complete empirical research matrix comparing vulnerability detection efficacy across human-authored and AI-generated code cohorts:

| Experiment ID | Training Dataset | Evaluation Benchmark | Tested Hypothesis |
| :--- | :--- | :--- | :--- |
| **E1** | **Devign Train (21,854)** | **Official Devign Test (2,732)** | Human In-Distribution Performance Benchmark |
| **E2** | **Devign Train (21,854)** | **LLMVul Holdout (4,120)** | Cross-Domain Zero-Shot Transfer to AI Code |
| **E3** | **Devign + LLMVul Train (36,485)** | **LLMVul Holdout (4,120)** | Mixed Dual-Domain Transfer & Domain Robustness |
| **E4** | **LLMVul Train (14,631)** | **LLMVul Holdout (4,120)** | AI In-Domain Vulnerability Detection Benchmark |
| **Supplementary** | **Devign Train** | **SeCodePLT Eval (2,480)** | Zero-Shot Transfer to Synthetic CVE/Patch Tasks |

### Rigorous Methodological Guarantees:
- **Zero Repository Leakage:** The LLMVul Holdout partition contains 8 repositories with strictly 0% overlap with LLMVul Train.
- **Quarantine of Uncertain Labels:** 2,679 ambiguous/tool-disagreement samples are excluded from binary evaluation.
- **Gold Standard Validation:** Evaluated against 100 manually reviewed, dual-expert consensus samples.

### Step 1: Environment Setup & Google Drive Mounting

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive/SecureCode-AI')
    models_dir = drive_root / 'models'
    metrics_dir = drive_root / 'metrics'
    figures_dir = drive_root / 'figures'
    print(f'[OK] Drive mounted. Artifacts loaded from: {drive_root}')
except ImportError:
    drive_root = Path('results')
    models_dir = drive_root / 'models'
    metrics_dir = drive_root / 'metrics'
    figures_dir = drive_root / 'figures'
    print(f'[OK] Running locally. Artifacts loaded from: {drive_root}')

### Step 2: Clone Repository & Install Dependencies

In [ ]:
from pathlib import Path

if Path('nlp').exists():
    %cd nlp
    !git pull origin main
elif Path('datasets').exists():
    !git pull origin main
else:
    !git clone https://github.com/0Tusharbhardwaj/nlp.git
    %cd nlp

!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn pyarrow joblib
print('[OK] Dependencies installed.')

### Step 3: Load All Evaluation Cohorts (Devign, LLMVul, SeCodePLT)

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Verify datasets exist
if not Path('datasets/processed/llmvul_test_clean.jsonl').exists():
    print('[!] Processed benchmarks not detected. Running downloader and partitioner...')
    !python scripts/download_full_datasets.py
    !python src/data/llmvul_partitioner.py
    !python -c "from src.data.secodeplt_adapter import SeCodePLTAdapter; SeCodePLTAdapter().export_evaluation_jsonl(Path('datasets/processed/secodeplt_eval.jsonl'))"

def read_jsonl(p):
    with open(p, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

# 1. Devign Test Split (Dataset A)
devign_test_df = pd.read_parquet('datasets/raw/full/devign/test.parquet')
e1_data = [{'code': row['func'], 'target': int(row['target']), 'id': f'devign_{row["id"]}'} for _, row in devign_test_df.iterrows()]

# 2. LLMVul Holdout Split (Dataset B - 8 disjoint repos)
e2_data = read_jsonl('datasets/processed/llmvul_test_clean.jsonl')

# 3. LLMVul Manual Review Gold Cohort (100 samples)
gold_raw = read_jsonl('datasets/processed/llmvul_manual_gold.jsonl')
gold_data = [x for x in gold_raw if x['target'] is not None]

# 4. SeCodePLT Eval Pairs (Dataset C)
secode_data = read_jsonl('datasets/processed/secodeplt_eval.jsonl')

print(f'[OK] Devign Test Cohort (E1):          {len(e1_data):,} samples | Vuln Rate: {sum(x["target"] for x in e1_data)/len(e1_data):.2%}')
print(f'[OK] LLMVul Holdout Cohort (E2/E3/E4): {len(e2_data):,} samples | Vuln Rate: {sum(x["target"] for x in e2_data)/len(e2_data):.2%}')
print(f'[OK] LLMVul Expert Gold Standard:       {len(gold_data):,} samples | Vuln Rate: {sum(x["target"] for x in gold_data)/len(gold_data):.2%}')
print(f'[OK] SeCodePLT Benchmark:               {len(secode_data):,} samples | Vuln Rate: {sum(x["target"] for x in secode_data)/len(secode_data):.2%}')

### Step 4: Model Ingestion (CodeBERT & Classical Baselines)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {device}')

# Path to Devign-trained CodeBERT model
devign_model_path = models_dir / 'codebert_devign_full'
if not devign_model_path.exists():
    print(f'[INFO] Fine-tuned model not found at {devign_model_path}. Using base checkpoint for demonstration.')
    devign_model_path = 'microsoft/codebert-base'

tokenizer = AutoTokenizer.from_pretrained(str(devign_model_path))
devign_codebert = AutoModelForSequenceClassification.from_pretrained(str(devign_model_path), num_labels=2).to(device)
devign_codebert.eval()
print(f'[OK] Devign-trained CodeBERT loaded from: {devign_model_path}')

# Check for E3 (Mixed) and E4 (LLMVul) models if already trained
mixed_model_path = models_dir / 'codebert_mixed_full'
llmvul_model_path = models_dir / 'codebert_llmvul_full'
mixed_codebert = None
llmvul_codebert = None

if mixed_model_path.exists():
    mixed_codebert = AutoModelForSequenceClassification.from_pretrained(str(mixed_model_path), num_labels=2).to(device)
    mixed_codebert.eval()
    print(f'[OK] E3 Mixed CodeBERT loaded from: {mixed_model_path}')
else:
    print('[INFO] E3 Mixed Model checkpoint not found in Drive. Will evaluate when trained.')

if llmvul_model_path.exists():
    llmvul_codebert = AutoModelForSequenceClassification.from_pretrained(str(llmvul_model_path), num_labels=2).to(device)
    llmvul_codebert.eval()
    print(f'[OK] E4 LLMVul-specific CodeBERT loaded from: {llmvul_model_path}')
else:
    print('[INFO] E4 LLMVul Model checkpoint not found in Drive. Will evaluate when trained.')

### Step 5: Batched Inference and Metric Evaluation Harness

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np
from tqdm.auto import tqdm

def run_inference(model, tokenizer, samples, batch_size=32):
    all_probs = []
    all_preds = []
    all_targets = [x['target'] for x in samples]
    codes = [x['code'] for x in samples]
    
    for i in tqdm(range(0, len(codes), batch_size), desc='Inference'):
        batch_codes = codes[i:i+batch_size]
        encoded = tokenizer(
            batch_codes,
            truncation=True,
            max_length=512,
            padding=True,
            return_tensors='pt',
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds = np.argmax(logits.cpu().numpy(), axis=-1)
            all_probs.extend(probs)
            all_preds.extend(preds)
            
    cm = confusion_matrix(all_targets, all_preds)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, zero_division=0)
    rec = recall_score(all_targets, all_preds, zero_division=0)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    binary_f1 = f1_score(all_targets, all_preds, average='binary')
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    roc_auc = roc_auc_score(all_targets, all_probs) if len(set(all_targets)) > 1 else 0.5
    
    return {
        'accuracy': float(acc),
        'precision': float(prec),
        'recall': float(rec),
        'macro_f1': float(macro_f1),
        'binary_f1': float(binary_f1),
        'fnr': float(fnr),
        'roc_auc': float(roc_auc),
        'confusion_matrix': cm.tolist(),
        'probabilities': [float(p) for p in all_probs],
        'predictions': [int(p) for p in all_preds],
    }

### Step 6: Execute Cross-Dataset Evaluations

1. **E1:** Devign Train -> Official Devign Test (2,732 samples)

2. **E2:** Devign Train -> LLMVul Holdout (4,120 samples from 8 disjoint repos)

3. **E3:** Devign + LLMVul Train -> LLMVul Holdout (Mixed Model)

4. **E4:** LLMVul Train -> LLMVul Holdout (AI In-Domain Model)

5. **Gold:** LLMVul Manual Review Gold Cohort (85 consensus samples)

6. **Supplementary:** SeCodePLT Benchmark (2,480 samples)

In [ ]:
print('\n--- Running E1: Devign Test Set (2,732 samples) ---')
e1_results = run_inference(devign_codebert, tokenizer, e1_data)

print('\n--- Running E2: Devign Model -> LLMVul Holdout (4,120 samples) ---')
e2_results = run_inference(devign_codebert, tokenizer, e2_data)

e3_results = None
if mixed_codebert is not None:
    print('\n--- Running E3: Mixed Model -> LLMVul Holdout (4,120 samples) ---')
    e3_results = run_inference(mixed_codebert, tokenizer, e2_data)

e4_results = None
if llmvul_codebert is not None:
    print('\n--- Running E4: LLMVul-Trained Model -> LLMVul Holdout (4,120 samples) ---')
    e4_results = run_inference(llmvul_codebert, tokenizer, e2_data)

print('\n--- Running Gold Standard: LLMVul Expert Review (85 consensus samples) ---')
gold_results = run_inference(devign_codebert, tokenizer, gold_data)

print('\n--- Running Supplementary: SeCodePLT Benchmark (2,480 samples) ---')
secode_results = run_inference(devign_codebert, tokenizer, secode_data)

### Step 7: Domain Drop Analysis & Comprehensive Performance Table

In [ ]:
import pandas as pd

delta_f1 = e2_results['macro_f1'] - e1_results['macro_f1']
delta_fnr = e2_results['fnr'] - e1_results['fnr']

summary_data = [
    {
        'Experiment': 'E1: Devign Train -> Devign Test',
        'Cohort Type': 'Human Legacy In-Distribution',
        'Samples': len(e1_data),
        'Accuracy': f"{e1_results['accuracy']:.4f}",
        'Precision': f"{e1_results['precision']:.4f}",
        'Recall': f"{e1_results['recall']:.4f}",
        'Macro F1': f"{e1_results['macro_f1']:.4f}",
        'FNR': f"{e1_results['fnr']:.4f}",
        'ROC-AUC': f"{e1_results['roc_auc']:.4f}",
    },
    {
        'Experiment': 'E2: Devign Train -> LLMVul Holdout',
        'Cohort Type': 'Zero-Shot Cross-Domain (AI Code)',
        'Samples': len(e2_data),
        'Accuracy': f"{e2_results['accuracy']:.4f}",
        'Precision': f"{e2_results['precision']:.4f}",
        'Recall': f"{e2_results['recall']:.4f}",
        'Macro F1': f"{e2_results['macro_f1']:.4f}",
        'FNR': f"{e2_results['fnr']:.4f}",
        'ROC-AUC': f"{e2_results['roc_auc']:.4f}",
    },
]

if e3_results is not None:
    summary_data.append({
        'Experiment': 'E3: Devign + LLMVul Train -> LLMVul Holdout',
        'Cohort Type': 'Mixed-Domain Dual Training',
        'Samples': len(e2_data),
        'Accuracy': f"{e3_results['accuracy']:.4f}",
        'Precision': f"{e3_results['precision']:.4f}",
        'Recall': f"{e3_results['recall']:.4f}",
        'Macro F1': f"{e3_results['macro_f1']:.4f}",
        'FNR': f"{e3_results['fnr']:.4f}",
        'ROC-AUC': f"{e3_results['roc_auc']:.4f}",
    })

if e4_results is not None:
    summary_data.append({
        'Experiment': 'E4: LLMVul Train -> LLMVul Holdout',
        'Cohort Type': 'AI In-Domain Specific Training',
        'Samples': len(e2_data),
        'Accuracy': f"{e4_results['accuracy']:.4f}",
        'Precision': f"{e4_results['precision']:.4f}",
        'Recall': f"{e4_results['recall']:.4f}",
        'Macro F1': f"{e4_results['macro_f1']:.4f}",
        'FNR': f"{e4_results['fnr']:.4f}",
        'ROC-AUC': f"{e4_results['roc_auc']:.4f}",
    })

summary_data.extend([
    {
        'Experiment': 'LLMVul Expert Gold Standard',
        'Cohort Type': 'Human-Audited AI Code',
        'Samples': len(gold_data),
        'Accuracy': f"{gold_results['accuracy']:.4f}",
        'Precision': f"{gold_results['precision']:.4f}",
        'Recall': f"{gold_results['recall']:.4f}",
        'Macro F1': f"{gold_results['macro_f1']:.4f}",
        'FNR': f"{gold_results['fnr']:.4f}",
        'ROC-AUC': f"{gold_results['roc_auc']:.4f}",
    },
    {
        'Experiment': 'Supplementary: SeCodePLT',
        'Cohort Type': 'Synthetic CVE/Patch Tasks',
        'Samples': len(secode_data),
        'Accuracy': f"{secode_results['accuracy']:.4f}",
        'Precision': f"{secode_results['precision']:.4f}",
        'Recall': f"{secode_results['recall']:.4f}",
        'Macro F1': f"{secode_results['macro_f1']:.4f}",
        'FNR': f"{secode_results['fnr']:.4f}",
        'ROC-AUC': f"{secode_results['roc_auc']:.4f}",
    },
])

summary_df = pd.DataFrame(summary_data)
print('\n' + '='*80)
print(' SECURECODE AI — CROSS-DATASET EVALUATION BENCHMARK MATRIX')
print('='*80)
print(summary_df.to_string(index=False))
print('='*80)
print(f'\n[Domain Shift Finding]:')
print(f'  Delta Macro F1 (E2 - E1): {delta_f1:+.4f}')
print(f'  Delta FNR (E2 - E1):      {delta_fnr:+.4f}')

### Step 8: Visualization & Experiment Artifact Export

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot side-by-side confusion matrices for E1 and E2
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(e1_results['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Safe', 'Vuln'], yticklabels=['Safe', 'Vuln'])
axes[0].set_title(f'E1: Devign In-Distribution\n(Macro F1: {e1_results["macro_f1"]:.3f} | FNR: {e1_results["fnr"]:.2%})')
axes[0].set_ylabel('Ground Truth')
axes[0].set_xlabel('Predicted')

sns.heatmap(e2_results['confusion_matrix'], annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Safe', 'Vuln'], yticklabels=['Safe', 'Vuln'])
axes[1].set_title(f'E2: LLMVul Zero-Shot (AI Code)\n(Macro F1: {e2_results["macro_f1"]:.3f} | FNR: {e2_results["fnr"]:.2%})')
axes[1].set_ylabel('Ground Truth')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
cm_fig_path = figures_dir / 'cross_dataset_e1_vs_e2_cm.png'
plt.savefig(cm_fig_path, dpi=300)
plt.show()
print(f'[OK] Confusion matrix figure saved to: {cm_fig_path}')

# Export results JSON
export_json_path = metrics_dir / 'cross_dataset_full_scale_results.json'
with open(export_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'e1_devign_test': {k: v for k, v in e1_results.items() if k not in ['probabilities', 'predictions']},
        'e2_llmvul_holdout': {k: v for k, v in e2_results.items() if k not in ['probabilities', 'predictions']},
        'gold_standard': {k: v for k, v in gold_results.items() if k not in ['probabilities', 'predictions']},
        'secodeplt_transfer': {k: v for k, v in secode_results.items() if k not in ['probabilities', 'predictions']},
        'delta_metrics': {
            'delta_macro_f1': float(delta_f1),
            'delta_fnr': float(delta_fnr),
        },
    }, f, indent=2)
print(f'[OK] Comprehensive results JSON exported to: {export_json_path}')